In [1]:
#| default_exp maggi.utils

In [28]:
#| export
import os, torch, json, torch.multiprocessing as mp, joblib, numpy as np, scipy.sparse as sp, argparse, math, re
import torch.nn.functional as F

from typing import Optional
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoConfig

from xcai.metrics import *
from xcai.sdata import SMainXCDataset, SXCDataset, identity_collate_fn

from sugar.core import load_raw_file

## `Tokenize`

In [3]:
#| export
DATASETS = {
    "arguana": "ArguAna",
    "msmarco": "MSMARCO",
    "climate-fever": "ClimateFEVER",
    "dbpedia-entity": "DBPedia",
    "fever": "FEVER",
    "fiqa": "FiQA2018",
    "hotpotqa": "HotpotQA",
    "nfcorpus": "NFCorpus",
    "nq": "NQ",
    "quora": "QuoraRetrieval",
    "scidocs": "SCIDOCS",
    "scifact": "SciFact",
    "webis-touche2020": "Touche2020",
    "trec-covid": "TRECCOVID",
    "musique": "MuSiQue",
}

In [6]:
#| export
def get_instruction(fname:str, dset:str):
    with open(fname) as file:
        instructions = json.load(file)
    assert dset in instructions, f"Dataset not present in the list: {dset}"
    return instructions[dset]
    

In [40]:
#| export
def tokenized_labels(lbl_info, idx:int, parts:int, model_name:str, max_length:Optional[int]=512):
    if isinstance(lbl_info, str):
        lbl_ids, lbl_txt = load_raw_file(lbl_info)
    elif isinstance(lbl_info, tuple) and len(lbl_info) == 2:
        lbl_ids, lbl_txt = lbl_info
    else:
        lbl_ids, lbl_txt = list(range(len(lbl_info))), lbl_info

    num_lbls = len(lbl_ids)
    bsize = math.ceil(num_lbls / parts)
    start_idx, end_idx = idx*bsize, (idx + 1)*bsize

    lbl_ids, lbl_txt = lbl_ids[start_idx:end_idx], lbl_txt[start_idx:end_idx]

    tokz = AutoTokenizer.from_pretrained(model_name)
    tokz.padding_side = "right"

    lbl_toks = tokz(lbl_txt, padding=True, return_tensors='pt', truncation=True, max_length=max_length)

    lbl_info = {'input_text': lbl_txt, 'identifier': lbl_ids}
    lbl_info.update(lbl_toks)

    dataset = SXCDataset(SMainXCDataset(data_info=lbl_info))
    return dataset
    

In [42]:
#| export
def tokenized_query(qry_info, idx:int, parts:int, instruction:str, dset_name:str, model_name:str, 
                    max_length:Optional[int]=512):
    
    if isinstance(qry_info, str):
        qry_ids, qry_txt = load_raw_file(qry_info)
    elif isinstance(qry_info, tuple) and len(qry_info) == 2:
        qry_ids, qry_txt = qry_info
    else:
        qry_ids, qry_txt = list(range(len(qry_info))), qry_info

    num_qrys = len(qry_ids)
    bsize = math.ceil(num_qrys / parts)
    start_idx, end_idx = idx*bsize, (idx + 1)*bsize

    instruction = get_instruction(instruction, DATASETS[dset_name])["query"] if os.path.exists(instruction) else instruction
    data_prompt_func = lambda x: "Instruct: " + instruction + f" Query: {x}"

    qry_ids, qry_txt = qry_ids[start_idx:end_idx], qry_txt[start_idx:end_idx]
    qry_txt = [data_prompt_func(o) for o in qry_txt]

    tokz = AutoTokenizer.from_pretrained(model_name)
    tokz.padding_side = "right"

    qry_toks = tokz(qry_txt, padding=True, return_tensors='pt', truncation=True, max_length=max_length)
    instruct_len = len(tokz(data_prompt_func(''))["input_ids"]) - 1

    qry_info = {'input_text': qry_txt, 'identifier': qry_ids}
    qry_info.update(qry_toks)

    qry_info["pool_mask"] = qry_info["attention_mask"].clone()
    qry_info["pool_mask"][:, :instruct_len] = 0

    dataset = SXCDataset(SMainXCDataset(data_info=qry_info))
    return dataset
    

## `Representation`

In [44]:
#| export
def get_and_save_representation(learn, dataset, fname:Optional[str]=None):
    if fname is not None and os.path.exists(fname):
        return torch.load(fname)
    rep = learn._get_data_representation(dataset)
    rep = rep.to(torch.float16)
    if fname is not None: torch.save(rep, fname)
    return rep
    

In [45]:
#| export
def _get_num_parts(dirname:str, fname:str):
    parts = [int(re.match(fname + r"00-([0-9]{2}).pth", o)[1]) for o in os.listdir(dirname) if re.match(fname + r"00-([0-9]{2}).pth", o)]
    assert len(parts) == 1
    return parts[0]

def combine_embeddings(fname:str, role:str, suffix:Optional[str]=None):
    suffix = "_repr_" if suffix is None else f"_repr{suffix}_"
    suffix = role + suffix

    if os.path.exists(fname):
        rep = torch.load(fname)
    else:
        output_dir = os.path.dirname(fname)
        num_parts = _get_num_parts(output_dir, suffix)
        rep = torch.vstack([torch.load(f'{output_dir}/{suffix}{idx:02d}-{num_parts:02d}.pth') for idx in range(num_parts)])
        torch.save(rep, fname)
    return rep
    

## `Compute metric`

In [47]:
#| export
def compute_metrics(data_repr:sp.csr_matrix, lbl_repr:sp.csr_matrix, data_mat:Optional[sp.csr_matrix]=None, data_batch_sz:Optional[int]=1000,
                    lbl_batch_sz:Optional[int]=10000, metric_type:Optional[str]="M"):
    if lbl_repr.shape[1] == data_repr.shape[1]:
        lbl_repr = lbl_repr.T

    metric = None
    if data_mat is not None:
        if metric_type == "M":
            metric = PrecReclMrr(lbl_repr.shape[1], pk=10, rk=200, rep_pk=[1, 3, 5, 10], rep_rk=[10, 100, 200], mk=[5, 10, 20])
        elif metric_type == "H":
            metric = PrecReclHits(lbl_repr.shape[1], pk=10, rk=200, hk=10, rep_pk=[1, 3, 5, 10], rep_rk=[5, 10, 100, 200], rep_hk=[1, 3, 5, 10])
        else:
            raise ValueError(f"Invalid metric type: {metric_type}")

    if metric is not None: metric.reset()

    all_data, all_indices, all_indptr = [], [], []

    for i in tqdm(range(0, data_repr.shape[0], data_batch_sz)):
        rep = data_repr[i:i+data_batch_sz].to('cuda')
        if metric is not None: gt = data_mat[i:i+data_batch_sz]

        # Compute scores
        scores = []
        for j in tqdm(range(0, lbl_repr.shape[1], lbl_batch_sz)):
            sc = rep@lbl_repr[:, j:j+lbl_batch_sz].to('cuda')
            scores.append(sc.to('cpu'))
        scores = torch.hstack(scores)

        # Top-k scores
        scores, indices = torch.topk(scores, k=200, dim=1, largest=True)
        o = {
            'pred_score': scores.flatten().to(torch.float32),
            'pred_idx': indices.flatten().to(torch.float32),
            'pred_ptr': torch.full((len(rep),), 200, dtype=torch.int64),
        }

        if metric is not None:
            t = {
                'targ_idx': torch.tensor(gt.indices, dtype=torch.int64),
                'targ_score': torch.tensor(gt.data, dtype=torch.float32),
                'targ_ptr': torch.tensor([p-q for p,q in zip(gt.indptr[1:], gt.indptr)], dtype=torch.int64),
            }
            metric.accumulate(**o, **t)

        all_data.append(o["pred_score"])
        all_indices.append(o["pred_idx"])
        all_indptr.append(o["pred_ptr"])

    all_data = torch.hstack(all_data)
    all_indices = torch.hstack(all_indices)
    all_indptr = torch.hstack([torch.zeros(1, dtype=torch.long), torch.hstack(all_indptr).cumsum(dim=0)])

    pred_lbl = sp.csr_matrix((all_data, all_indices, all_indptr), shape=(data_repr.shape[0], lbl_repr.shape[1]), dtype=np.float32)

    return metric if metric is None else {k:float(v) for k,v in metric.value.items()}, pred_lbl
    